<a href="https://colab.research.google.com/github/jabri62018/Jabri_lab/blob/Jabri_lab/Zx_Spacetime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# file= Zx_Spacetime.ipynb
# Author= Eng. Abdulla Al-Jabri
# Zero input: Spacetime constants from Zx

import mpmath as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
from IPython.display import display, HTML

print("="*60)
print("PROBLEM: Reproduce spacetime constants from Zx zeros")
print("Target: t_P, l_P, h, G, c")
print("="*60)

mp.mp.dps = 80
xp = 21.0

def Zx(t):
    x = mp.mpc('0.5', str(t))
    return mp.exp(-x/xp) * mp.exp(-5*mp.log(x)) * mp.log(x) * mp.sin(2*mp.pi/x)

def find_zeros(n=20):
    mp.mp.dps = 60
    t_vals = np.arange(14.0, 120, 0.02)
    f_vals = [float(mp.im(Zx(t))) for t in t_vals]
    brackets = []
    for j in range(len(f_vals)-1):
        if f_vals[j] * f_vals[j+1] < 0:
            brackets.append((t_vals[j], t_vals[j+1]))
    mp.mp.dps = 80
    zeros = []
    for t_min, t_max in brackets[:n]:
        r = mp.findroot(lambda tt: mp.im(Zx(tt)), (t_min, t_max), tol=mp.mpf('1e-65'))
        zeros.append(float(r))
    return zeros

zeros = find_zeros(20)

CONSTANTS = {
    't_P': 5.391e-44, 'l_P': 1.616e-35,
    'h': 6.62607015e-34, 'G': 6.67430e-11, 'c': 299792458,
}

def calc_C(gamma):
    h = mp.mpf('1e-15')
    t = mp.mpf(gamma)
    z = Zx(t)
    zppp = (Zx(t+2*h) - 2*Zx(t+h) + 2*Zx(t-h) - Zx(t-2*h)) / (2*h)
    return float(0.5 * gamma**2 * mp.re(zppp / z))

rows = []
available = CONSTANTS.copy()
for i, g in enumerate(zeros, 1):
    C = calc_C(g)
    logC = np.log10(abs(C) + 1e-300)
    diffs = {k: abs(logC - np.log10(abs(v) + 1e-300)) for k,v in available.items()}
    if not diffs: break
    matched = min(diffs, key=diffs.get)
    rows.append({'Root': i, 'gamma': g, 'C_calc': C, 'Matched': matched,
                 'Value': available[matched], 'Log Diff': diffs[matched]})
    del available[matched]

df = pd.DataFrame(rows)
df.to_csv('Zx_Spacetime_match.csv', index=False, float_format='%.15e')

plt.figure(figsize=(10,6))
plt.loglog(df['C_calc'], df['gamma'], 'o-', markersize=8, label='Zx roots')
for _, r in df.iterrows():
    plt.annotate(r['Matched'], (r['C_calc'], r['gamma']), fontsize=11, weight='bold')
plt.axvline(CONSTANTS['t_P'], color='red', linestyle='--', label='Planck time')
plt.axvline(CONSTANTS['l_P'], color='green', linestyle='--', label='Planck length')
plt.xlabel('C_calc')
plt.ylabel('gamma zero')
plt.title('Zx Spacetime Matching')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.savefig('Zx_Spacetime_plot.png', dpi=300)
plt.close()

print("\n=== TABLE ===")
display(df[['Root','gamma','C_calc','Matched','Log Diff']])
display(HTML('<img src="Zx_Spacetime_plot.png" width="800">'))
print("\n=== SUMMARY ===")
for _, r in df.iterrows():
    print(f"Root {r['Root']:2d}: {r['Matched']:5s} | Log Diff = {r['Log Diff']:.2e}")

with zipfile.ZipFile('Zx_Spacetime_results.zip', 'w') as zipf:
    zipf.write('Zx_Spacetime_match.csv')
    zipf.write('Zx_Spacetime_plot.png')
from google.colab import files
files.download('Zx_Spacetime_results.zip')
print("\nDone. Files downloaded.")

PROBLEM: Reproduce spacetime constants from Zx zeros
Target: t_P, l_P, h, G, c


/tmp/ipykernel_6039/831138039.py:72: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  plt.axvline(CONSTANTS['t_P'], color='red', linestyle='--', label='Planck time')



=== TABLE ===


,Root,gamma,C_calc,Matched,Log Diff
0,1,15.053925,-9.367783e-30,h,4.150381
1,2,74.140130,-4.959475e-31,l_P,4.486994



=== SUMMARY ===
Root  1: h     | Log Diff = 4.15e+00
Root  2: l_P   | Log Diff = 4.49e+00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done. Files downloaded.
